In [0]:
adding a bew ine

In [0]:
%sql
CREATE OR REPLACE TABLE Employees (
    EmployeeID INT,
    Name STRING,
    DepartmentID INT,
    Salary INT
);

INSERT INTO Employees (EmployeeID, Name, DepartmentID, Salary) VALUES
(1, 'Alice', 1, 70000),
(2, 'Bob', 1, 85000),
(3, 'Charlie', 2, 75000),
(4, 'Diana', 2, 75000),
(5, 'Evan', 3, 90000),
(6, 'Fiona', 3, 85000),
(7, 'George', 1, 60000),
(8, 'Hannah', 2, 70000);


CREATE OR REPLACE TABLE Departments (
    DepartmentID INT,
    DepartmentName STRING
);

INSERT INTO Departments (DepartmentID, DepartmentName) VALUES
(1, 'HR'),
(2, 'Finance'),
(3, 'Engineering');


In [0]:
%sql
select name, salary, dense_rank() over (order by salary desc) rnk from employees 

In [0]:
%sql
select * from (select e.name, e.salary, d.departmentname,  rank() over (partition by d.departmentName order by e.salary desc ) rnk from employees e join departments d on e.departmentid = d.departmentid) where rnk = 1

In [0]:
%sql
select * from departments

In [0]:
%sql
select * from employees

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Create a sample dataset
data = [
    ("Alice", "Sales", "2023-01-10", 1200),
    ("Alice", "Sales", "2023-02-15", 1500),
    ("Alice", "Sales", "2023-03-10", 1600),
    ("Bob", "Sales", "2023-01-12", 1300),
    ("Bob", "Sales", "2023-02-16", 1400),
    ("Bob", "Sales", "2023-03-18", 1700),
    ("Carol", "Marketing", "2023-01-11", 1800),
    ("Carol", "Marketing", "2023-02-14", 1900),
    ("Carol", "Marketing", "2023-03-12", 2000),
    ("Dave", "Marketing", "2023-01-10", 1000),
    ("Dave", "Marketing", "2023-02-15", 1100),
    ("Dave", "Marketing", "2023-03-11", 1050)
]

columns = ["employee_name", "department", "sale_date", "sales"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Convert date column to date type
df = df.withColumn("sale_date", to_date("sale_date", "yyyy-MM-dd"))

# Show DataFrame
df.show()

In [0]:
spec = Window.partitionBy("department").orderBy(col("sales").desc())

df = df.withColumn("rnk", rank().over(spec))

df.display()

In [0]:
spec = Window.partitionBy("employee_name").orderBy("sale_date")
df = df.withColumn("dffrnce", col("sales")-lag("sales",1).over(spec))
df.display()

In [0]:
window_spec = Window.partitionBy("employee_name").orderBy("sale_date")\
                    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df.withColumn("running_avg", avg("sales").over(window_spec)).show()

In [0]:
df1.alias("target").merge(df2.alias("source"), "target.id == source.id and target.iscurrent is null)\
    .whenMatchedUpdate(
        condition = t.isdate != s.isdate,
        set = {
            "end_date" : "current_date",
            "is_current" : "false"
        }
        )\
            .whenNotMatchedInsertAll(
                values{

                }
            ).execute()

In [0]:
%sql
CREATE OR REPLACE TABLE employees (
    emp_id INT,
    emp_name STRING,
    manager_id INT,
    department STRING,
    salary INT
);

INSERT INTO employees VALUES
(1,  'Alice',    NULL,      'Executive',  30000),
(2,  'Bob',      1,         'Engineering',12000),
(3,  'Charlie',  1,         'Engineering',11000),
(4,  'David',    2,         'Engineering',9000),
(5,  'Eva',      2,         'Engineering',9500),
(6,  'Frank',    3,         'Engineering',8500),
(7,  'Grace',    NULL,      'HR',         10000),
(8,  'Hannah',   7,         'HR',         8000),
(9,  'Ivy',      7,         'HR',         7500);


In [0]:
%sql
SELECT DISTINCT e.emp_id, e.emp_name
FROM employees e
JOIN employees m
  ON e.emp_id = m.manager_id;


In [0]:
%sql
select emp_name from employees where manager_id is null

In [0]:
%sql
select e.emp_name employeename , m.emp_name as managername from employees e left join employees m on e.manager_id = m.emp_id    where m.emp_id = 2

In [0]:
%sql
select * from employees 

#### PIVOT Functions 

In [0]:
# from pyspark.sql import SparkSession
from pyspark.sql import Row

# spark = SparkSession.builder.appName("PivotExamples").getOrCreate()

data = [
    Row(region="North", product="A", sales=100),
    Row(region="North", product="B", sales=150),
    Row(region="South", product="A", sales=200),
    Row(region="South", product="B", sales=250),
    Row(region="East", product="A", sales=300),
    Row(region="East", product="B", sales=350),
    Row(region="West", product="A", sales=400),
    Row(region="West", product="B", sales=450),
]

df = spark.createDataFrame(data)
df.show()


In [0]:
df.groupBy("product").pivot("region").sum("sales").display()